In [1]:
import pandas as pd

df = pd.read_csv("guardian_articles.csv")

theresa_may_mask = (
    (df["sectionName"] == "Opinion") &
    (df["webTitle"].str.contains(r"theresa may|\btheresa\b", case=False, na=False)) &
    (df["webTitle"].str.contains("brexit", case=False, na=False))
)

boris_johnson_mask = (
    (df["sectionName"] == "Opinion") &
    (df["webTitle"].str.contains(r"boris johnson|\bboris\b", case=False, na=False)) &
    (df["webTitle"].str.contains("brexit", case=False, na=False))
)

boris_johnson_filtered_df = df.loc[boris_johnson_mask].copy()
print(f"Articles found: {len(boris_johnson_filtered_df)}")
boris_johnson_filtered_df["word_count"] = boris_johnson_filtered_df["bodyContent"].str.split().str.len()

theresa_may_filtered_df = df.loc[theresa_may_mask].copy()
print(f"Articles found: {len(theresa_may_filtered_df)}")
theresa_may_filtered_df["word_count"] = theresa_may_filtered_df["bodyContent"].str.split().str.len()


theresa_may_articles = theresa_may_filtered_df["bodyContent"]
boris_johnson_articles = boris_johnson_filtered_df["bodyContent"]
theresa_may_articles = theresa_may_articles.fillna("").astype(str)
boris_johnson_articles = boris_johnson_articles.fillna("").astype(str)






Articles found: 26
Articles found: 31


In [2]:
import re
import spacy
from collections import defaultdict

nlp = spacy.load("en_coreference_web_trf")

# Explicit-name patterns (controls which clusters we keep)
TARGETS = {
    "Theresa May": re.compile(r"\btheresa\s+may\b", re.I),
    "Boris Johnson": re.compile(r"\bboris\s+johnson\b", re.I),
}

def entity_snippets(text, sent_window=0, targets=TARGETS):
    """
    Returns: {"Theresa May": [snippet,...], "Boris Johnson": [snippet,...]}
    Includes pronouns because we take the whole coref cluster,
    but we ONLY keep clusters that contain an explicit name mention.
    Dedupes by sentence-window span to avoid near-identical repeats.
    """
    doc = nlp(text)

    # coref clusters stored as span groups
    clusters = [v for k, v in doc.spans.items() if k.startswith("coref_clusters_")]
    sents = list(doc.sents)

    # token index -> sentence index
    tok2sent = {}
    for si, s in enumerate(sents):
        for t in range(s.start, s.end):
            tok2sent[t] = si

    out = {canon: [] for canon in targets}
    seen = {canon: set() for canon in targets}  # dedupe by (lo,hi)

    for spangroup in clusters:
        if not spangroup:
            continue

        # Which target(s) does this cluster belong to?
        hit_targets = [
            canon for canon, pat in targets.items()
            if any(pat.search(m.text) for m in spangroup)
        ]
        if not hit_targets:
            continue

        # Collect snippets for ALL mentions in the cluster (incl pronouns)
        for mention in spangroup:
            si = tok2sent.get(mention.start)
            if si is None:
                continue

            lo = max(0, si - sent_window)
            hi = min(len(sents), si + sent_window + 1)
            snippet = " ".join(s.text.strip() for s in sents[lo:hi]).strip()

            for canon in hit_targets:
                span_key = (lo, hi)
                if span_key in seen[canon]:
                    continue
                seen[canon].add(span_key)
                out[canon].append(snippet)

    return out

# --- DROP-IN usage: same structure as your loops ---

for i in range(len(theresa_may_articles)):
    body_text = theresa_may_articles.iloc[i]
    snippets_by_entity = entity_snippets(body_text, sent_window=0)
    snippets = snippets_by_entity.get("Theresa May", [])
    if snippets:
        print("\nEntity: Theresa May")
        for j, snippet in enumerate(snippets[:3], 1):
            print(f"  {j}. {snippet}")

for i in range(len(boris_johnson_articles)):
    print(f"\n--- Article {i+1} ---")
    body_text = boris_johnson_articles.iloc[i]
    snippets_by_entity = entity_snippets(body_text, sent_window=0)
    snippets = snippets_by_entity.get("Boris Johnson", [])
    if snippets:
        print("\nEntity: Boris Johnson")
        for j, snippet in enumerate(snippets[:3], 1):
            print(f"  {j}. {snippet}")

import pandas as pd

rows = []

# --- Theresa May ---
for article_idx, body_text in enumerate(theresa_may_articles):
    snippets_by_entity = entity_snippets(body_text, sent_window=1)
    snippets = snippets_by_entity.get("Theresa May", [])

    for snip_idx, snippet in enumerate(snippets, 1):
        rows.append({
            "person": "Theresa May",
            "article_index": article_idx,
            "snippet_index": snip_idx,
            "snippet": snippet,
        })

# --- Boris Johnson ---
for article_idx, body_text in enumerate(boris_johnson_articles):
    snippets_by_entity = entity_snippets(body_text, sent_window=1)
    snippets = snippets_by_entity.get("Boris Johnson", [])

    for snip_idx, snippet in enumerate(snippets, 1):
        rows.append({
            "person": "Boris Johnson",
            "article_index": article_idx,
            "snippet_index": snip_idx,
            "snippet": snippet,
        })

df_out = pd.DataFrame(rows)

df_out[df_out["person"] == "Theresa May"] \
    .to_csv("theresa_may_snippets.csv", index=False)

df_out[df_out["person"] == "Boris Johnson"] \
    .to_csv("boris_johnson_snippets.csv", index=False)


/Users/ameliemajor/anaconda3/envs/bertopic/lib/python3.11/site-packages/spacy/cli/info.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/ameliemajor/anaconda3/envs/bertopic/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/ameliemajor/anaconda3/envs/bertopic/lib/python3.11/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):



Entity: Theresa May
  1. On Wednesday Theresa May sent the country a message about authority.
  2. Summoning the cabinet to Chequers and then summoning the television cameras to record her opening statement, she was saying unmistakably: “Keep calm and remember who’s boss.”
  3. But the pith of her short message – no covert attempt to stay inside the EU, no second referendum – made clear which side she is coming down on, despite No 10’s “motherhood and apple pie” briefing that Britain’s future relationship with Europe will involve controls on immigration and be good for trade.

Entity: Theresa May
  1. Theresa May freewheeled through the opening weeks of her premiership.
  2. She was fortunate to enjoy a combination of a summer recess, post-referendum political fatigue and feelgood Olympic distraction.
  3. Decent economic numbers, Labour divisions and the Tory party rallying behind her have helped too.

Entity: Theresa May
  1. For all the (largely male) characterisation of Theresa Ma